In [1]:

# Install the libraries required for our RAG system

!pip -q install pymupdf sentence-transformers chromadb google-genai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 969.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [2]:

from google.colab import files

uploaded = files.upload()


Saving HandBook_b9c6eSTUDENT HANDBOOK 2025.pdf to HandBook_b9c6eSTUDENT HANDBOOK 2025.pdf


In [3]:

import os

pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]

if not pdf_files:
    raise ValueError("No PDF uploaded. Please upload the Student Handbook.")

PDF_PATH = pdf_files[0]

print("Handbook uploaded successfully!")
print("File name:", PDF_PATH)
print("File size:", round(os.path.getsize(PDF_PATH) / (1024 * 1024), 2), "MB")


Handbook uploaded successfully!
File name: HandBook_b9c6eSTUDENT HANDBOOK 2025.pdf
File size: 2.78 MB


In [4]:

import fitz  # PyMuPDF

doc = fitz.open(PDF_PATH)

pages = []

for page_index, page in enumerate(doc):
    page_text = page.get_text("text").strip()

    if page_text:
        pages.append({
            "page": page_index + 1,
            "text": page_text
        })

print("Total PDF pages:", len(doc))
print("Pages containing extractable text:", len(pages))
print("\nSample text from the first page:\n")
print(pages[0]["text"][:1500] if pages else "No text extracted.")


Total PDF pages: 69
Pages containing extractable text: 69

Sample text from the first page:

STUDENT
HANDBOOK
(For Guidance & Compliance by Students)
2025-2026


In [5]:

import pymupdf

doc = pymupdf.open(PDF_PATH)

pages = []

for page_index, page in enumerate(doc):
    page_text = page.get_text("text").strip()

    if page_text:
        pages.append({
            "page": page_index + 1,
            "text": page_text
        })

print("Total PDF pages:", len(doc))
print("Pages with extractable text:", len(pages))


Total PDF pages: 69
Pages with extractable text: 69


In [6]:

# Split each page's text into chunks while retaining page metadata

CHUNK_SIZE = 180       # words per chunk
CHUNK_OVERLAP = 40     # overlapping words between chunks

chunks = []

for page in pages:
    words = page["text"].split()
    start = 0

    while start < len(words):
        end = min(start + CHUNK_SIZE, len(words))
        chunk_text = " ".join(words[start:end])

        if chunk_text.strip():
            chunks.append({
                "text": chunk_text,
                "page": page["page"]
            })

        if end == len(words):
            break

        start = end - CHUNK_OVERLAP

print("Total chunks created:", len(chunks))
print("\nFirst chunk metadata:")
print("PDF page:", chunks[0]["page"])
print("\nFirst chunk text:\n")
print(chunks[0]["text"][:1000])


Total chunks created: 154

First chunk metadata:
PDF page: 1

First chunk text:

STUDENT HANDBOOK (For Guidance & Compliance by Students) 2025-2026


In [7]:

from sentence_transformers import SentenceTransformer

# Load a lightweight sentence embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract the text from each chunk
chunk_texts = [chunk["text"] for chunk in chunks]

# Convert text into numerical vectors
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", embeddings.shape)
print("First embedding (first 10 values):")
print(embeddings[0][:10])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Number of chunks: 154
Embedding shape: (154, 384)
First embedding (first 10 values):
[-0.01124261  0.05674429  0.03492983 -0.02852871 -0.00112595  0.01360415
 -0.08105284 -0.00436979 -0.09757398  0.04387843]


In [8]:

import chromadb

# Create the vector database
client = chromadb.Client()

# Create or access the handbook collection
collection = client.get_or_create_collection(
    name="student_handbook",
    metadata={"hnsw:space": "cosine"}
)

# Store chunks, embeddings, and their original PDF page numbers
collection.upsert(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunk_texts,
    embeddings=embeddings.tolist(),
    metadatas=[
        {"page": chunk["page"]}
        for chunk in chunks
    ]
)

print("Total chunks stored:", collection.count())
print("Vector database created successfully!")


Total chunks stored: 154
Vector database created successfully!


In [9]:

question = "What is the minimum attendance required to appear in the End Semester Examination?"

# Convert the question into an embedding
question_embedding = embedding_model.encode(question).tolist()

# Retrieve the three most relevant chunks
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for i, (text, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n--- Retrieved Chunk {i} ---")
    print("PDF page:", metadata["page"])
    print("Distance:", round(distance, 4))
    print("Context:", text[:1000])




--- Retrieved Chunk 1 ---
PDF page: 19
Distance: 0.3271
Context: will not be permitted to appear in the End Semester Examination (ESE) unless, under extreme and special circumstances, the student is permitted by the Hon’ble Vice Chancellor by condoning attendance up to 5% in a course with B+ cap on Grade. • However, Attendance Rules of relevant Regulatory Bodies/AUUP will be applicable whichever is higher. • Minimum Attendance criteria applicable on the programmes under Regulatory bodies is as under: Regulatory Body Minimum Attendance Criteria to appear in the End Semester Examination RCI 80% Course Work, 90% for Practicum & 100% in School Internship PCI 80% NCTE 80% Normal & 90% in School Internship BCI 75% CoA 75% subject to student is passed in Continuous Internal Assessment IAP (Physiotherapy Association) 75% AIOTA (Occupational Therapy Association) 75%

--- Retrieved Chunk 2 ---
PDF page: 40
Distance: 0.3964
Context: 38 performance. Summer semester is conducted for a maximum of 1

In [10]:

from getpass import getpass
from google import genai

api_key = getpass("Enter your Gemini API key: ")

gemini_client = genai.Client(api_key=api_key)

print("Gemini client configured successfully!")


Enter your Gemini API key: ··········
Gemini client configured successfully!


In [17]:

import time
from google.genai import errors

def ask_handbook(question, top_k=3):
    # 1. Embed the question
    question_embedding = embedding_model.encode(question).tolist()

    # 2. Retrieve relevant handbook chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = results["documents"][0]
    metadata_list = results["metadatas"][0]
    distances = results["distances"][0]

    # 3. Build context with PDF page numbers
    context_parts = []

    for text, metadata in zip(retrieved_chunks, metadata_list):
        context_parts.append(
            f"[PDF page {metadata['page']}]\n{text}"
        )

    context = "\n\n".join(context_parts)

    # 4. Build the prompt INSIDE the function
    prompt = f"""
You are a university Student Handbook Assistant.

Answer using ONLY the handbook context below.

Rules:
- Do not invent policies or details.
- Preserve important conditions and exceptions.
- Cite relevant PDF page numbers.
- If the context is insufficient, say so clearly.
- Do not guess.

HANDBOOK CONTEXT:
{context}

STUDENT QUESTION:
{question}

ANSWER:
"""

    # 5. Generate an answer, retrying temporary server errors
    response = None

    for attempt in range(3):
        try:
            response = gemini_client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )
            break

        except errors.ServerError as e:
            if attempt == 2:
                print("Gemini server is still unavailable.")
                raise

            wait_seconds = 2 ** (attempt + 1)
            print(f"Temporary server error. Retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)

    if response is None or not response.text:
        raise RuntimeError("Gemini returned no answer. Please try again.")

    # 6. Return the results
    return {
        "question": question,
        "context": context,
        "answer": response.text,
        "sources": sorted(set(m["page"] for m in metadata_list)),
        "distances": distances
    }

print("Handbook question-answering function created!")


Handbook question-answering function created!


In [18]:

result = ask_handbook(
    "What is the minimum attendance required to appear in the End Semester Examination?"
)

print("STUDENT QUESTION")
print(result["question"])

print("\nRETRIEVED CONTEXT")
print(result["context"])

print("\nGENERATED ANSWER")
print(result["answer"])

print("\nSOURCE PDF PAGES")
print(result["sources"])


STUDENT QUESTION
What is the minimum attendance required to appear in the End Semester Examination?

RETRIEVED CONTEXT
[PDF page 19]
will not be permitted to appear in the End Semester Examination (ESE) unless, under extreme and special circumstances, the student is permitted by the Hon’ble Vice Chancellor by condoning attendance up to 5% in a course with B+ cap on Grade. • However, Attendance Rules of relevant Regulatory Bodies/AUUP will be applicable whichever is higher. • Minimum Attendance criteria applicable on the programmes under Regulatory bodies is as under: Regulatory Body Minimum Attendance Criteria to appear in the End Semester Examination RCI 80% Course Work, 90% for Practicum & 100% in School Internship PCI 80% NCTE 80% Normal & 90% in School Internship BCI 75% CoA 75% subject to student is passed in Continuous Internal Assessment IAP (Physiotherapy Association) 75% AIOTA (Occupational Therapy Association) 75%

[PDF page 40]
38 performance. Summer semester is conducted fo

In [19]:

test_questions = [
    "What is the minimum attendance required to appear in the End Semester Examination?",
    "What actions are considered unfair means during examinations?",
    "What grades and grade points are used in the university grading system?",
    "How can a student appeal or request a review of an examination punishment?",
    "What is the maximum duration allowed to complete an undergraduate programme?"
]

evaluation_results = []

for i, question in enumerate(test_questions, start=1):
    print("\n" + "=" * 80)
    print(f"QUESTION {i}: {question}")

    try:
        result = ask_handbook(question)

        print("\nRETRIEVED CONTEXT:")
        print(result["context"])

        print("\nGENERATED ANSWER:")
        print(result["answer"])

        print("\nSOURCE PDF PAGES:")
        print(result["sources"])

        evaluation_results.append(result)

    except Exception as e:
        print("Error:", str(e))
        print("You can retry this question after checking the error.")



QUESTION 1: What is the minimum attendance required to appear in the End Semester Examination?
Temporary server error. Retrying in 2s...
Temporary server error. Retrying in 4s...
Gemini server is still unavailable.
Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
You can retry this question after checking the error.

QUESTION 2: What actions are considered unfair means during examinations?
Temporary server error. Retrying in 2s...

RETRIEVED CONTEXT:
[PDF page 24]
22 • Disturbing or disrupting or instigating others to disturb/disrupt the examination. • Instigating others to leave the examination room. • Carrying any weapons into the examination centre. • Carrying photocopy or scanned copy of admit card. • Any act not specified above as determined by the Academic Council. 3.11.2 Acts of Unfair Means The following shall be deemed to be 

In [20]:

question = "What is the minimum attendance required to appear in the End Semester Examination?"

question_embedding = embedding_model.encode(question).tolist()

results = collection.query(
    query_embeddings=[question_embedding],
    n_results=6,
    include=["documents", "metadatas", "distances"]
)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n--- Result {i} ---")
    print(f"PDF page: {metadata['page']}")
    print(f"Distance: {distance:.4f}")
    print(document)



--- Result 1 ---
PDF page: 19
Distance: 0.3271
will not be permitted to appear in the End Semester Examination (ESE) unless, under extreme and special circumstances, the student is permitted by the Hon’ble Vice Chancellor by condoning attendance up to 5% in a course with B+ cap on Grade. • However, Attendance Rules of relevant Regulatory Bodies/AUUP will be applicable whichever is higher. • Minimum Attendance criteria applicable on the programmes under Regulatory bodies is as under: Regulatory Body Minimum Attendance Criteria to appear in the End Semester Examination RCI 80% Course Work, 90% for Practicum & 100% in School Internship PCI 80% NCTE 80% Normal & 90% in School Internship BCI 75% CoA 75% subject to student is passed in Continuous Internal Assessment IAP (Physiotherapy Association) 75% AIOTA (Occupational Therapy Association) 75%

--- Result 2 ---
PDF page: 40
Distance: 0.3964
38 performance. Summer semester is conducted for a maximum of 15 credit units and up to 05 courses.

In [21]:

response = None

for attempt in range(5):
    try:
        response = gemini_client.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt
        )
        break

    except errors.ServerError:
        if attempt == 4:
            print("Gemini is still unavailable. Please try again later.")
            raise

        wait_seconds = 2 ** (attempt + 1)
        print(f"Temporary server error. Retrying in {wait_seconds}s...")
        time.sleep(wait_seconds)

if response is None or not response.text:
    raise RuntimeError("Gemini returned no answer. Please try again.")


NameError: name 'prompt' is not defined

In [22]:

import time
from google.genai import errors

test_prompt = """
Answer in one sentence: What is 2 + 2?
"""

response = None

for attempt in range(5):
    try:
        response = gemini_client.models.generate_content(
            model="gemini-3.8-flash",
            contents=test_prompt
        )
        break
    except errors.ServerError:
        if attempt == 4:
            print("Gemini is unavailable. Try again later.")
            raise
        wait_seconds = 2 ** (attempt + 1)
        print(f"Retrying in {wait_seconds} seconds...")
        time.sleep(wait_seconds)

if response and response.text:
    print(response.text)


2 + 2 equals 4.


In [23]:
result = ask_handbook(
    "What is the minimum attendance required to appear in the End Semester Examination?",
    top_k=6
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCE PDF PAGES:")
print(result["sources"])

Temporary server error. Retrying in 2s...
Temporary server error. Retrying in 4s...
ANSWER:
Based on the handbook context, the minimum attendance requirements to appear in the End Semester Examination (ESE) are as follows:

### General Requirement
* A student whose attendance is **less than 75%**, regardless of the reason for the shortfall, will **not** be permitted to appear in the End Semester Examination [PDF page 19].
* **Relaxation:** A maximum relaxation of 25% attendance may be allowed for sickness (upon submission of a valid medical certificate) or other valid reasons beyond the student's control, provided written permission from the HoI/HoD is obtained [PDF page 19].
* **Condonation Exception:** Under extreme and special circumstances, the Hon’ble Vice Chancellor may permit a student to appear by condoning attendance up to 5% in a course, with a "B+" cap on the Grade [PDF page 19].

---

### Programmes Under Regulatory Bodies
The attendance rules of relevant Regulatory Bodies/

In [24]:

prompt = f"""
You are an assistant for the Amity University Student Handbook 2025-2026.

Answer the student's question using ONLY the handbook context provided.

Rules:
1. Answer the exact question asked.
2. Include all relevant general rules and programme-specific rules.
3. Preserve exceptions, conditions, and deadlines.
4. Do not assume that a rule applies to every programme if the handbook lists exceptions.
5. Cite the PDF page number next to each relevant rule.
6. If the retrieved context is incomplete, clearly say so.
7. Do not list unrelated information.

HANDBOOK CONTEXT:
{context}

STUDENT QUESTION:
{question}

ANSWER:
"""


NameError: name 'context' is not defined

In [25]:
result = ask_handbook(
    "How can a student appeal an examination punishment?",
    top_k=3
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCE PDF PAGES:")
print(result["sources"])

Temporary server error. Retrying in 2s...
QUESTION:
How can a student appeal an examination punishment?

ANSWER:
Based on the handbook context, a student can appeal an examination punishment through the following process:

* **Filing the Appeal/Representation:** A student on whom any punishment has been imposed may make a representation for a review of their case to the **Vice Chancellor** within **30 days** from the date of receipt of the communication regarding the punishment [PDF page 26].
* **Review Process:** 
  * If deemed necessary, the Vice Chancellor may refer the case back to the Examination Discipline Committee (EDC) for review [PDF page 26].
  * The EDC's recommendations on the appeal will be placed before the Vice Chancellor, who will then review the case and pass orders as considered fit, or refer it to the Academic Council for advice [PDF page 26].
* **Special Provision for Expulsion:** If a student has been expelled from the University under these regulations, the Acade